***Negative sampling + interaction matrix***

The objective of this notebook is to expand the dataset (interactions), by using negative sampling. 
For every transaction a customer makes 4 non purchases are of items not bought by user. This will make a total label ratio of 4:1.


In [1]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

df = pd.read_csv('data_processed/cleaned_retail_data.csv')

le_user = LabelEncoder()
le_item = LabelEncoder()

df['user_id'] = le_user.fit_transform(df['Customer ID'])
df['item_id'] = le_item.fit_transform(df['StockCode'])
df.to_csv('data_processed/cleaned_retail_data.csv', index=False)

In [2]:
NUM_USERS = df['user_id'].nunique()
NUM_ITEMS = df['item_id'].nunique()

In [3]:
interactions = df[['user_id', 'item_id']]
interactions = interactions.drop_duplicates()
interactions['purchased'] = 1
interactions = interactions.sort_values(by='user_id').reset_index(drop=True)
interactions.head()

,user_id,item_id,purchased
0,0,3104,1
1,0,1834,1
2,0,1212,1
3,0,219,1
4,0,1562,1


*Creating the negative samples*

In [4]:
import numpy as np
# 1. Create a fast-lookup map: {user_id: {set of item_ids they bought}}
# This eliminates the need for your `check_if_purchased` function entirely!
user_history = interactions.groupby('user_id')['item_id'].apply(set).to_dict()

# # We need the unique pool of user_ids from the original rows to match your loop
original_users = interactions['user_id'].values

negative_samples = []
num_negative_samples = 4

# 2. Generate negative samples efficiently
for user_id in original_users:
    # Get the items this user has already bought (default to empty set if not found)
    already_purchased = user_history.get(user_id, set())
    
    added = 0
    while added < num_negative_samples:
        random_item = np.random.randint(0, NUM_ITEMS)
        
        # Fast O(1) check using Python sets instead of searching a pandas DataFrame
        if random_item not in already_purchased:
            negative_samples.append({
                'user_id': user_id,
                'item_id': random_item,
                'purchased': 0
            })
            added += 1

# 3. Turn the list into a DataFrame and combine it with the original interactions
df_negatives = pd.DataFrame(negative_samples)
interactions = pd.concat([interactions, df_negatives], ignore_index=True)

In [5]:
print(interactions.shape)
print(interactions['purchased'].value_counts())
interactions.head()

(2409660, 3)
purchased
0    1927728
1     481932
Name: count, dtype: int64


,user_id,item_id,purchased
0,0,3104,1
1,0,1834,1
2,0,1212,1
3,0,219,1
4,0,1562,1


In [6]:
rfm = pd.read_csv('data_processed/rfm_clustered.csv')

In [7]:
rfm_clusters = rfm[['Customer ID', 'Cluster']].copy()

rfm_clusters = rfm_clusters.rename(columns={'Cluster': 'cluster_id'})
rfm_clusters['user_id'] = le_user.transform(rfm_clusters['Customer ID'])

rfm['user_id'] = le_user.transform(rfm['Customer ID'])
rfm.to_csv('data_processed/rfm_clustered.csv', index=False)
rfm_clusters
# You need to map Customer ID to user_id using your LabelEncoder
# How would you do this? — just this one thing figure out yourself

,Customer ID,cluster_id,user_id
0,12346,3,0
1,12347,3,1
2,12348,2,2
3,12349,2,3
4,12350,0,4
...,...,...,...
5873,18283,3,5873
5874,18284,0,5874
5875,18285,0,5875
5876,18286,0,5876


In [8]:
print(f"Unique customers in df: {df['Customer ID'].nunique()}")
print(f"Classes in le_user: {len(le_user.classes_)}")

print(f"Unique items in df: {df['StockCode'].nunique()}")
print(f"Classes in le_item: {len(le_item.classes_)}")

Unique customers in df: 5878
Classes in le_user: 5878
Unique items in df: 4631
Classes in le_item: 4631


In [9]:
interactions = interactions.merge(rfm_clusters[['user_id', 'cluster_id']], on='user_id', how='left')

In [10]:
interactions.to_csv('data_processed/interactions.csv', index=False)
interactions

,user_id,item_id,purchased,cluster_id
0,0,3104,1,3
1,0,1834,1,3
2,0,1212,1,3
3,0,219,1,3
4,0,1562,1,3
...,...,...,...,...
2409655,5877,1490,0,2
2409656,5877,3525,0,2
2409657,5877,1744,0,2
2409658,5877,928,0,2
